# 03 — Pipeline de RAG: construção e avaliação

**Tech Challenge Fase 3 — MedFlow AI**

Cada seção deste notebook responde a uma **pergunta de pesquisa** explícita, seguida de método,
resultado, interpretação, trade-off e limitação — a estrutura exigida pelo feedback da Fase 1.

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd()
while not (RAIZ / "src" / "medflow_ai").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
print("Raiz do projeto:", RAIZ)

## 1. Como o corpus vira chunks recuperáveis?

**Método.** Document Loaders do LangChain → uma unidade por seção `##` → `RecursiveCharacterTextSplitter`
→ `chunk_id` estável por conteúdo.

In [ ]:
from medflow_ai.rag.loaders import load_protocol_documents
from medflow_ai.rag.chunking import chunk_documents

secoes = load_protocol_documents()
chunks = chunk_documents(secoes, chunk_size=400, chunk_overlap=100)
print(f"{len(secoes)} seções → {len(chunks)} chunks")

exemplo = chunks[0]
print("\nMetadados de rastreabilidade de um chunk:")
for chave, valor in exemplo.metadata.items():
    print(f"  {chave:16s}: {valor}")

**Interpretação.** Todo chunk carrega `doc_id`, `section_id`, `chunk_id`, versão e vigência do
documento. Sem isso, "citar a fonte" seria apenas escrever um nome de arquivo no fim da resposta.

## 2. Qual backend de embedding usar?

**Pergunta.** É possível medir RAG de forma reprodutível sem depender de download de modelo?

**Método.** O backend padrão é um *hashing trick* determinístico (n-gramas de palavra e de caractere,
TF sublinear, normalização L2). Um backend denso (`sentence-transformers`) fica disponível como opção.

In [ ]:
from medflow_ai.rag.embeddings import get_embeddings
import numpy as np

emb = get_embeddings("hashing")
print("backend:", emb.name)

consulta = "quando repetir o TSH após ajustar a dose de levotiroxina"
textos = [
    "O controle do TSH deve ser feito 6 a 8 semanas após início ou ajuste de dose.",
    "A radiografia de tórax não é rotina na crise asmática.",
    "Coletar hemoculturas antes do antimicrobiano na sepse.",
]
q = np.array(emb.embed_query(consulta))
for texto, vetor in zip(textos, emb.embed_documents(textos)):
    print(f"  cos={float(np.array(vetor) @ q):+.3f}  {texto[:70]}")

**Trade-off registrado.** O backend por hashing é essencialmente lexical: ele não captura sinonímia
profunda. Em compensação, é determinístico, roda em CI sem GPU e permite que qualquer pessoa regenere
as métricas do relatório com um comando. A avaliação da seção 4 mede exatamente o custo dessa escolha.

## 3. Como o índice é construído e persistido?

In [ ]:
from medflow_ai.rag.vector_store import MedFlowVectorStore

store = MedFlowVectorStore.from_documents(chunks, emb)
print(f"{len(store)} chunks indexados")

for doc, score in store.similarity_search_with_score("valores críticos comunicados pelo laboratório", k=3):
    print(f"  {score:.3f}  {doc.metadata['citation'][:88]}")

## 4. Qual configuração de recuperação é a melhor? (experimento principal)

**Pergunta de pesquisa.** Dadas 40 perguntas clínicas com seção-ouro conhecida, qual combinação de
estratégia × `k` × tamanho de chunk recupera o trecho certo com mais frequência e em posição mais alta?

**Método.** Grade completa de 4 estratégias × 3 valores de `k` × 3 tamanhos de chunk = 36 configurações.
Métricas: `hit@k` (seção correta entre os k primeiros), `doc_hit@k` (documento correto) e `MRR`.

In [ ]:
from medflow_ai.evaluation.rag_eval import run_experiment_grid, save_results, load_benchmark

benchmark = load_benchmark()
print(f"{len(benchmark)} perguntas com gabarito")
print("Exemplo:", benchmark[0].question, "→", benchmark[0].gold_section_id)

resultados = run_experiment_grid()
caminhos = save_results(resultados)
print(f"\n{len(resultados)} configurações avaliadas · artefatos em {caminhos['csv']}")

In [ ]:
import pandas as pd
tabela = pd.DataFrame([{k: v for k, v in r.to_dict().items() if k != "falhas"} for r in resultados])
tabela.sort_values(["hit_at_k", "mrr"], ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

pivo = tabela[tabela.chunk_size == 400].pivot(index="k", columns="strategy", values="hit_at_k")
eixo = pivo.plot(marker="o", figsize=(8, 4.5))
eixo.set_title("Pergunta: qual estratégia recupera a seção correta com mais frequência? (chunk=400)")
eixo.set_xlabel("k (trechos recuperados)"); eixo.set_ylabel("hit@k")
eixo.set_xticks(sorted(tabela.k.unique())); eixo.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
eixo = tabela.pivot_table(index="chunk_size", columns="strategy", values="mrr").plot(
    kind="bar", figsize=(8, 4.5))
eixo.set_title("Pergunta: o tamanho do chunk muda a posição do acerto? (MRR médio)")
eixo.set_ylabel("MRR"); eixo.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

**Resultado (execução registrada em `artifacts/rag/rag_experiments.csv`).**

- `hybrid`, `k=5`, `chunk=400` obteve o melhor `hit@5`;
- `doc_hit@5` fica próximo de 1,0 em quase todas as configurações — o sistema quase sempre acerta o
  **documento**, e erra com mais frequência a **seção** dentro dele;
- `mmr` degrada nitidamente com chunks maiores.

**Interpretação.** A penalização de redundância do MMR é contraproducente neste corpus: várias seções
do mesmo protocolo são legitimamente relevantes para a mesma pergunta, e o MMR as afasta em favor de
diversidade artificial. Por isso a configuração de produção adotou `hybrid`.

**Trade-off.** `hybrid` custa cerca de 3–4× a latência da busca densa pura (fusão de dois rankings).
Na escala deste corpus isso é irrelevante (sub-milissegundo); em um corpus hospitalar real, mereceria
nova medição.

**Limitação declarada.** O benchmark é construído sobre o **mesmo corpus** indexado. Ele mede
qualidade de recuperação, **não** generalização para documentos inéditos. As perguntas foram escritas
com vocabulário diferente do texto-fonte para reduzir casamento lexical trivial, mas isso não elimina
a limitação.

## 5. Onde o RAG erra?

In [ ]:
from medflow_ai.rag.retriever import ProtocolRetriever
from medflow_ai.evaluation.rag_eval import evaluate_retriever

recuperador = ProtocolRetriever(store, strategy="hybrid", k=5)
metricas = evaluate_retriever(recuperador, benchmark, k=5)
print(f"hit@5={metricas.hit_at_k} doc_hit@5={metricas.doc_hit_at_k} MRR={metricas.mrr}")
print("Perguntas com falha:", metricas.falhas)

por_id = {item.id: item for item in benchmark}
for falha in metricas.falhas[:4]:
    item = por_id[falha]
    print(f"\n[{item.id}] {item.question}")
    print(f"  esperado : {item.gold_section_id}")
    for chunk in recuperador.retrieve(item.question, k=3):
        print(f"  obtido   : {chunk.section_id:20s} score={chunk.score:.3f}")

**Interpretação dos erros.** As falhas concentram-se em perguntas cujo vocabulário se afasta do
texto-fonte (paráfrase forte) — exatamente o ponto fraco previsto para um embedding lexical. É o caso
em que um embedding denso multilíngue tende a ajudar; a troca é de uma linha
(`MEDFLOW_EMBEDDING_BACKEND=sentence_transformers`) e está documentada no README.

## 6. O RAG melhora a resposta final? (ablação)

In [ ]:
from medflow_ai.llm.prompts import build_messages, format_protocol_block
from medflow_ai.llm.providers import get_chat_model
from medflow_ai.fine_tuning.evaluate import compare_systems

modelo = get_chat_model("template")

def com_rag(pergunta):
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context=format_protocol_block(recuperador.retrieve(pergunta)),
                               safety_status="SAFE")
    return str(modelo.invoke(mensagens).content)

def sem_rag(pergunta):
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context="Nenhum trecho de protocolo foi recuperado.",
                               safety_status="SAFE")
    return str(modelo.invoke(mensagens).content)

comparacao = compare_systems({"sem_rag": sem_rag, "com_rag": com_rag})
pd.DataFrame([{k: v for k, v in r.to_dict().items() if k != "exemplos"} for r in comparacao])

**Interpretação.** Sem RAG, o gerador não tem o que citar: `citacao_presente` e `groundedness`
caem a zero e o sistema corretamente declara falta de evidência em vez de inventar. Com RAG, todas as
respostas citam fonte e a maioria cita o documento correto.

**Limitação.** O gerador usado nesta ablação é o baseline extrativo determinístico, não uma LLM. A
mesma medição roda no notebook 02 com o modelo base e o fine-tuned.